# Mã hoá lại 177.321 keyframe bằng encoder mạnh hơn## Vì sao làm việc nàyBTC cấp sẵn vector **ViT-B-32** — bản nhỏ nhất của họ CLIP. Đo trên bộ 81 query 2026,chỉ cần thay bằng **ViT-B-16** (nhỉnh hơn đúng một bậc) đã được:| | R@1 | R@5 | R@20 | R@50 | R@100 | FINAL ||---|---|---|---|---|---|---|| B-32 của BTC | 0.235 | 0.395 | 0.506 | 0.593 | 0.654 | 0.4765 || B-16 chấm lại rổ top-100 | 0.235 | 0.457 | 0.568 | 0.642 | 0.654 | **0.5111** || B-32 + 2·B-16 (trộn z) | 0.284 | 0.432 | 0.580 | 0.642 | 0.654 | **0.5185** |Và con số đó còn **bị nén**: rổ vẫn do B-32 bốc nên R@100 khoá cứng ở 0.654.Mã hoá lại toàn bộ thì rổ cũng tốt lên, kết quả thật sẽ cao hơn.Cả hai đội trong hai bài báo cùng dataset (AIO_Owlgorithms dùng BEiT-3, U-CESE dùngMobileCLIP) đều **không dùng vector BTC** mà tự mã hoá lại. Giờ ta có số của chính mình.## Điều quan trọng nhất: THỨ TỰ HÀNG`metadata.parquet` dòng *i* phải ứng với vector *i*. Lệch một dòng thì hệ thống vẫnchạy bình thường, chỉ là **trả về ảnh sai** — kiểu lỗi tệ nhất vì không ai nhận ra.Notebook này không tự suy ra thứ tự mà **đọc thẳng `metadata.parquet`** đi kèm trongdataset, rồi dựng đường dẫn ảnh từ `(video_id, n)`. Không có cửa nào lệch.## Chuẩn bị### Cách A — dùng dataset công khai (nên chọn, khỏi upload 12 GB)Keyframe đã có sẵn trên Kaggle: **`nguynnc/aic2025`**. Chỉ cần upload gói phụ 3 MB:```bashpython src/shrink_keyframes.py --side-onlycd data/kaggle_side_files && kaggle datasets create -p . -u```Gói phụ gồm `metadata.parquet`, `queries_hcmc2026.csv`, và `probe_b32.npz` —500 khung đối chứng để **chứng minh ảnh của họ đúng là ảnh của mình**.Add Data **cả hai** dataset.### Cách B — tự upload keyframe```bashpython src/shrink_keyframes.py --apply    # 28,66 GB -> 12,11 GB, 12 gói zipcd data/keyframes_kaggle_upload && kaggle datasets create -p . -u```Ảnh thu nhỏ cạnh ngắn còn 384px q92. **Đã đo là không làm tụt điểm** (B-16 từ ảnhgốc 0.5111 vs từ ảnh thu nhỏ 0.5111, dù cosine chỉ 0.9957 — bằng chứng cụ thể rằngcosine là chỉ số sai để quyết định việc này).> ⚠️ **Để dataset của mình ở chế độ PRIVATE** — nó chứa `queries_hcmc2026.csv`,> bộ eval tự annotate của đội. Đừng thêm `--public` (mặc định Kaggle vốn là private).## Cài đặt trên Kaggle* **Accelerator: GPU T4 × 2** (Settings → Accelerator)* **Internet: ON** (để tải trọng số model)* Add Data → dataset keyframe vừa upload

In [ ]:
!pip install -q open_clip_torch

## Cấu hình`MODELS` chạy tuần tự, mỗi model xuất một file vector riêng. Đo luôn Recall tại chỗnên **một lần chạy là biết model nào thắng**, khỏi tải về rồi mới đo.Ước tính trên T4×2 (177k ảnh):* `ViT-L-14/openai` — ~15 phút, 768 chiều, ảnh vào 224px* `ViT-B-16-SigLIP2-384/webli` — ~35 phút, 768 chiều, ảnh vào 384px* `ViT-L-16-SigLIP2-384/webli` — ~70 phút, 1024 chiều, ảnh vào 384pxVì sao có SigLIP2: dataset công khai `irongolemmc/hcmc-aic-2025-data` (một đội khácthi cùng giải) chứa sẵn `vectors_ViT-B-16-SigLIP2-384_webli.npy` — họ đã chọn đúngmodel này. Không phải bằng chứng nó tốt nhất, nhưng đủ để đưa vào danh sách thử.Họ cũng dùng `DINOv3-ViT-L/16` — model **chỉ nhìn ảnh, không đọc chữ**, nên đó làđể tìm-bằng-ảnh chứ không phải tìm-bằng-chữ. Việc riêng, chưa làm ở notebook này.

In [ ]:
from pathlib import Pathimport os, json, time# Quét TOÀN BỘ /kaggle/input chứ không cố định một slug: metadata nằm ở gói phụ# của mình còn ảnh có thể nằm ở dataset công khai của người khác. Gõ sai slug là# lỗi hay gặp nhất, và lần chạy Whisper trước đã dính (Kaggle gắn vào# /kaggle/input/datasets/<user>/<slug> chứ không phải /kaggle/input/<slug>).INPUT = Path("/kaggle/input")OUT_DIR = Path("/kaggle/working")# (tên model open_clip, trọng số). Bỏ bớt nếu sợ hết giờ — mỗi model độc lập,# chạy xong cái nào lưu cái đó nên đứt giữa chừng cũng không mất công.MODELS = [    ("ViT-L-14", "openai"),                  # nhanh, cùng họ với B-32 nên dễ so    ("ViT-B-16-SigLIP2-384", "webli"),       # đội khác cùng dataset chọn đúng cái này    ("ViT-L-16-SigLIP2-384", "webli"),       # bản to hơn của trên]BATCH      = 256      # giảm còn 128 nếu OOM (SigLIP-384 nặng gấp ~3 lần)WORKERS    = 4        # số tiến trình giải nén JPEGMAX_HOURS  = 10.5     # tự dừng trước trần 12h của KaggleN_EXPECTED = 177321if not INPUT.exists():    raise SystemExit("Không có /kaggle/input — chưa Add Data.")# MỘT lượt duyệt duy nhất, tìm cả thư mục ảnh lẫn ba file phụ.## Dùng os.walk chứ KHÔNG dùng rglob: rglob phải quét qua cả 177.321 FILE chỉ để# lọc ra ~873 thư mục. os.walk cho sửa thẳng danh sách thư mục con, nên ta CẮT# NHÁNH ngay khi gặp thư mục video — bên trong chỉ còn ảnh, không cần chui vào.IMG_EXT = (".jpg", ".jpeg", ".png", ".webp")def first_image_ext(path):    """Đuôi ảnh đầu tiên gặp trong thư mục, None nếu thư mục không chứa ảnh.    BẮT BUỘC phải lọc bằng cái này. Dataset AIC2025 có    `objects-aic25-b1/objects/L21_V001/001.json` — thư mục tên GIỐNG HỆT thư mục    ảnh, cũng có "_V". Nhận nhầm nó là thư mục video thì hoặc chết ở bước dò tên    file, hoặc tệ hơn là mã hoá nhầm. Và os.walk trên Linux KHÔNG duyệt theo thứ    tự chữ cái nên không thể trông chờ Keyframes_* được gặp trước.    Dừng ngay ở ảnh đầu tiên nên chỉ tốn một scandir cho mỗi thư mục ứng viên.    """    try:        with os.scandir(path) as it:            for e in it:                low = e.name.lower()                for x in IMG_EXT:                    if low.endswith(x):                        return x    except OSError:        pass    return NoneWANT = {"metadata.parquet", "queries_hcmc2026.csv", "probe_b32.npz"}VID_DIR, DUP, SIDE, SKIPPED = {}, [], {}, []t_scan = time.time()for root, dirs, files in os.walk(INPUT):    for f in WANT & set(files):        SIDE.setdefault(f, Path(root) / f)    keep = []    for name in dirs:        if "_V" not in name:            keep.append(name)            continue        p = Path(root) / name        if first_image_ext(p) is None:            SKIPPED.append(p)           # vd .../objects/L21_V001 toàn .json            continue                    # cũng không chui vào        if name in VID_DIR:            DUP.append(name)            # hai nguồn cùng chứa một video        else:            VID_DIR[name] = p    dirs[:] = keepmeta_path = SIDE.get("metadata.parquet")queries_path = SIDE.get("queries_hcmc2026.csv")if meta_path is None:    raise SystemExit("Không tìm thấy metadata.parquet. Đã Add gói phụ chưa?")print(f"quét xong trong {time.time()-t_scan:.1f}s")print(f"metadata: {meta_path}")print(f"queries : {queries_path}")print(f"probe   : {SIDE.get('probe_b32.npz')}")print(f"tìm thấy {len(VID_DIR)} thư mục CÓ ẢNH (mong đợi 873)")if SKIPPED:    print(f"bỏ qua {len(SKIPPED)} thư mục tên giống video nhưng không chứa ảnh "          f"(vd {SKIPPED[0]}) — đúng như mong đợi với objects-aic25-b1")if DUP:    # Không tự chọn hộ: hai bản của cùng một video có thể khác cách trích keyframe.    print(f"⚠ {len(DUP)} video xuất hiện ở NHIỀU dataset (vd {DUP[:3]}) — đang dùng "          f"bản gặp trước. Nếu ô kiểm tra căn hàng bên dưới báo hỏng thì bỏ bớt "          f"một dataset ở Add Data rồi chạy lại.")

## Dựng danh sách ảnh theo ĐÚNG thứ tự metadataĐây là ô quan trọng nhất của cả notebook. Mọi kiểm tra ở đây đều là `assert` —thà nổ ngay bây giờ còn hơn sinh ra bộ vector lệch hàng mà không ai biết.

In [ ]:
import pandas as pdmeta = pd.read_parquet(meta_path)print(f"{len(meta):,} hàng · cột: {list(meta.columns)}")assert len(meta) == N_EXPECTED, f"metadata có {len(meta)} hàng, mong đợi {N_EXPECTED}"# Bộ của mình đặt tên {n:03d}.jpg, nhưng dataset của người khác có thể đánh số# khác. Dò quy tắc MỘT lần trên video đầu tiên rồi áp cho tất cả, thay vì# đoán bừa và chết ở ảnh thứ 50.000.PATTERNS = ["{n:03d}.jpg", "{n:04d}.jpg", "{n}.jpg",            "{n:03d}.png", "{n:04d}.png", "{n:05d}.jpg", "{n:06d}.jpg"]v0 = meta.video_id.iloc[0]n0 = int(meta.n.iloc[0])FMT = next((p for p in PATTERNS if (VID_DIR[v0] / p.format(n=n0)).exists()), None)if FMT is None:    got = sorted(x.name for x in VID_DIR[v0].iterdir())[:5]    raise SystemExit(f"không đoán được cách đặt tên file. {v0} n={n0}, "                     f"trong thư mục có: {got}")print(f"quy tắc tên file: {FMT}")paths = [VID_DIR[v] / FMT.format(n=int(n)) for v, n in zip(meta.video_id, meta.n)]missing = [p for p in paths[::200] if not p.exists()]      # lấy mẫu cho nhanhassert not missing, f"thiếu ảnh, ví dụ: {missing[:3]}"# Kiểm tra kỹ hơn: đếm ảnh thật của từng video có khớp số hàng metadata không.# Quét MỘT lần rồi mới so — quét hai lần trên 873 thư mục là tự làm chậm mình.ext = "*" + FMT[FMT.rindex("."):]real = {v: sum(1 for _ in d.glob(ext)) for v, d in VID_DIR.items()}cnt = meta.groupby("video_id").size()bad = {v: (int(c), real.get(v, 0)) for v, c in cnt.items() if real.get(v, 0) != c}assert not bad, (f"lệch số ảnh ở {len(bad)} video (metadata, thực tế): "                 f"{list(bad.items())[:3]}")print(f"✅ {len(paths):,} đường dẫn dựng xong, khớp từng video")print(f"   đầu: {paths[0]}")print(f"   cuối: {paths[-1]}")

## 🔴 Kiểm tra CĂN HÀNG — chạy trước, hỏng thì dừng luônNếu keyframe lấy từ dataset công khai của người khác thì câu hỏi sống còn là:**ảnh của họ có đúng là ảnh của mình không?** Trích keyframe theo cách khác là sốthứ tự `n` lệch, và hệ thống sẽ trả về ảnh sai trong khi mọi thứ trông vẫn bình thường— kiểu lỗi tệ nhất vì không ai nhận ra.Cách chứng minh: mã hoá 500 khung ngẫu nhiên bằng đúng `ViT-B-32-quickgelu/openai`rồi so với vector BTC cấp. Trùng ảnh thì cosine ~**0.999**; lệch ảnh thì rơi thẳngxuống 0.2–0.5.Mốc đã đo sẵn ở máy để bạn đối chiếu:| nguồn ảnh | cosine TB | tệ nhất ||---|---|---|| ảnh gốc, đúng khung | **0.9999** | 0.9942 || ảnh thu nhỏ 384px, đúng khung | **0.9922** | 0.9410 || ảnh SAI khung | **0.5033** | 0.2148 |Ngưỡng đặt trên **trung bình** chứ không phải từng khung: với ảnh thu nhỏ vẫn cóvài khung tụt xuống 0.94 một cách bình thường, bắt lỗi theo từng khung là báo động giả.Khoảng cách giữa "đúng" (0.99+) và "sai" (0.50) rộng đến mức không thể nhầm.

In [ ]:
# Ô này cố ý TỰ ĐỦ: nó phải chạy được ngay cả khi bạn chỉ chạy riêng nó, và nó# đứng trước ô nạp model nên không mượn được biến ở dưới.import numpy as np, torch, open_clipfrom PIL import ImageDEVICE = "cuda" if torch.cuda.is_available() else "cpu"def check_alignment(thresh=0.95):    probe_path = SIDE.get("probe_b32.npz")    if probe_path is None:        print("⚠ không có probe_b32.npz — BỎ QUA kiểm tra căn hàng.")        print("  Chỉ chấp nhận được nếu keyframe là bộ mình tự upload.")        return None    z = np.load(probe_path)    pv, pn, pf = z["video_id"], z["n"], z["feat"]    print(f"đối chứng {len(pv)} khung từ {probe_path.name}")    model, _, prep = open_clip.create_model_and_transforms(        "ViT-B-32-quickgelu", pretrained="openai")    model = model.eval().to(DEVICE)    pp = [VID_DIR[v] / FMT.format(n=int(n)) for v, n in zip(pv, pn)]    got = np.zeros_like(pf)    with torch.no_grad():        for i in range(0, len(pp), 64):            ims = torch.stack([prep(Image.open(p).convert("RGB")) for p in pp[i:i+64]])            v = model.encode_image(ims.to(DEVICE)).float()            v /= v.norm(dim=-1, keepdim=True)            got[i:i+len(v)] = v.cpu().numpy()    del model    if DEVICE == "cuda":        torch.cuda.empty_cache()    cos = (got * pf).sum(1)    print(f"cosine với vector BTC: TB {cos.mean():.4f} · "          f"tệ nhất {cos.min():.4f} · dưới {thresh}: {(cos < thresh).sum()}/{len(cos)}")    if cos.mean() < thresh:        raise SystemExit(            f"❌ CĂN HÀNG SAI (cosine TB {cos.mean():.4f}). Keyframe của dataset này "            f"KHÔNG phải ảnh ứng với metadata.parquet. DỪNG — đừng mã hoá tiếp, "            f"vector sinh ra sẽ lệch hàng và hệ thống trả ảnh sai mà không báo gì.")    print("✅ căn hàng đúng — ảnh khớp metadata")    return float(cos.mean())ALIGN = check_alignment()

## Nạp ảnh`DataLoader` với nhiều tiến trình con vì **giải nén JPEG là nút thắt, không phải GPU**.Một T4 chạy ViT-L-14 fp16 ăn khoảng 150 ảnh/s, mà PIL giải một ảnh 384px mất ~3mstức ~330 ảnh/s một tiến trình — thiếu 2 GPU ăn nếu chỉ dùng luồng chính.

In [ ]:
import torchfrom torch.utils.data import Dataset, DataLoaderfrom PIL import Imageclass KeyframeSet(Dataset):    def __init__(self, paths, preprocess):        self.paths, self.prep = paths, preprocess    def __len__(self):        return len(self.paths)    def __getitem__(self, i):        return self.prep(Image.open(self.paths[i]).convert("RGB"))def make_loader(paths, preprocess, batch=BATCH, workers=WORKERS):    return DataLoader(KeyframeSet(paths, preprocess), batch_size=batch,                      num_workers=workers, pin_memory=True, shuffle=False,                      persistent_workers=False)DEVICE = "cuda" if torch.cuda.is_available() else "cpu"print(f"thiết bị: {DEVICE} · {torch.cuda.device_count()} GPU "      f"{[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}")if DEVICE == "cuda" and torch.cuda.device_count() < 2:    print("⚠ chỉ thấy 1 GPU — vào Settings → Accelerator chọn 'GPU T4 x2' để nhanh gấp đôi")

## Mã hoá`DataParallel` chia mỗi batch cho các GPU. Với suy luận thuần thì nó đơn giản và antoàn hơn tự quản luồng — không có trạng thái dùng chung nào để mà đua nhau.Lưu **float16**: vector đã L2-chuẩn hoá nên giá trị nằm trong [-1,1], float16 thừasức. Giảm nửa dung lượng (ViT-L-14: 519 MB → 260 MB), tải về nhanh gấp đôi.

In [ ]:
import numpy as np, open_clipclass ImageTower(torch.nn.Module):    """Bọc encode_image vào forward().    BẮT BUỘC phải có: DataParallel chỉ chia việc qua forward(). Gọi    net.module.encode_image(...) trông thì chạy nhưng thực ra chạy trên MỘT GPU    — mất trắng nửa số máy mà không có lấy một dòng cảnh báo.    """    def __init__(self, clip_model):        super().__init__()        self.clip = clip_model    def forward(self, x):        return self.clip.encode_image(x)def encode_all(model_name, pretrained):    tag = f"{model_name}__{pretrained}".replace("/", "_")    dst = OUT_DIR / f"features_{tag}.npy"    if dst.exists():        print(f"[{tag}] đã có {dst.name}, bỏ qua")        return dst    t0 = time.time()    model, _, prep = open_clip.create_model_and_transforms(model_name, pretrained=pretrained)    model = model.eval().half().to(DEVICE)    # Suy số chiều bằng một lượt chạy thật thay vì đọc thuộc tính: mỗi họ model    # để số chiều ở một chỗ khác nhau (visual.output_dim, text_projection...).    with torch.no_grad():        px = prep.transforms[0].size        px = px if isinstance(px, int) else px[0]        dim = model.encode_image(torch.zeros(1, 3, px, px).half().to(DEVICE)).shape[1]    print(f"[{tag}] nạp xong {time.time()-t0:.0f}s · {dim} chiều · ảnh vào {px}px")    net = ImageTower(model)    if torch.cuda.device_count() > 1:        net = torch.nn.DataParallel(net)    out = np.zeros((len(paths), dim), dtype=np.float16)    loader = make_loader(paths, prep)    done, t0 = 0, time.time()    with torch.no_grad():        for batch in loader:            v = net(batch.half().to(DEVICE, non_blocking=True)).float()            v /= v.norm(dim=-1, keepdim=True)            out[done:done + len(v)] = v.half().cpu().numpy()            done += len(v)            if done % (BATCH * 20) < BATCH:                el = time.time() - t0                print(f"  {done:,}/{len(paths):,} · {done/el:.0f} ảnh/s · "                      f"còn ~{(len(paths)-done)/(done/el)/60:.1f} phút", flush=True)            if (time.time() - t0) / 3600 > MAX_HOURS:                raise SystemExit("chạm trần thời gian, dừng để kịp lưu")    assert done == len(paths), f"mã hoá {done} ảnh nhưng cần {len(paths)}"    np.save(dst, out)    print(f"[{tag}] xong {(time.time()-t0)/60:.1f} phút -> {dst.name} "          f"({dst.stat().st_size/1024**2:.0f} MB)")    del model, net    if DEVICE == "cuda":        torch.cuda.empty_cache()    return dst

## Chấm điểm ngay tại chỗKhông đợi tải về mới biết. Dùng đúng 81 query đã annotate và đúng công thức`Final Score = trung bình R@{1,5,20,50,100}` của quy chế.Mốc cần vượt: **B-32 của BTC = 0.4765**.

In [ ]:
KS = (1, 5, 20, 50, 100)def evaluate(feat_path, model_name, pretrained):    if queries_path is None:        print("không có queries_hcmc2026.csv trong dataset — bỏ qua phần chấm điểm")        return None    q = pd.read_csv(queries_path)    feats = np.load(feat_path).astype(np.float32)    model, _, _ = open_clip.create_model_and_transforms(model_name, pretrained=pretrained)    tokz = open_clip.get_tokenizer(model_name)    model = model.eval().to(DEVICE)    with torch.no_grad():        tv = model.encode_text(tokz(q.text_en.tolist()).to(DEVICE)).float()        tv /= tv.norm(dim=-1, keepdim=True)    tv = tv.cpu().numpy()    del model    if DEVICE == "cuda":        torch.cuda.empty_cache()    vid, fx = meta.video_id.values, meta.frame_idx.values    ranks = []    F = torch.from_numpy(feats).to(DEVICE)    for i, row in enumerate(q.itertuples()):        s = (F @ torch.from_numpy(tv[i]).to(DEVICE)).cpu().numpy()        top = np.argpartition(-s, 100)[:100]        top = top[np.argsort(-s[top])]        ok = np.where((vid[top] == row.video_id) & (fx[top] >= row.frame_idx_min)                      & (fx[top] <= row.frame_idx_max))[0]        ranks.append(int(ok[0]) + 1 if len(ok) else None)    del F    if DEVICE == "cuda":        torch.cuda.empty_cache()    r = pd.Series(ranks, dtype="float")    vals = [(r <= k).sum() / len(q) for k in KS]    print("   " + "".join(f"R@{k:<7d}" for k in KS) + "FINAL")    print("   " + "".join(f"{v:<9.3f}" for v in vals) + f"{np.mean(vals):.4f}"          + ("   ✅ hơn B-32" if np.mean(vals) > 0.4765 else "   ❌ không hơn B-32"))    prog = pd.DataFrame({"p": q.video_id.str[:3], "hit": r.notna()})    print("   theo chương trình:",          dict(prog.groupby("p").hit.sum().astype(int)))    return float(np.mean(vals))

## Chạy

In [ ]:
results = {}for name, pre in MODELS:    print(f"\n{'='*70}\n{name} / {pre}\n{'='*70}", flush=True)    try:        p = encode_all(name, pre)        results[f"{name}/{pre}"] = evaluate(p, name, pre)    except Exception as e:        print(f"❌ {type(e).__name__}: {e}")        results[f"{name}/{pre}"] = Noneprint(f"\n\n{'='*70}\nTỔNG KẾT (nền B-32 của BTC = 0.4765)\n{'='*70}")for k, v in sorted(results.items(), key=lambda x: -(x[1] or 0)):    print(f"  {k:42s} {'hỏng' if v is None else f'{v:.4f}'}")

## Lưu manifest rồi tải vềGhi kèm tên model + trọng số vào manifest: bộ 2023 chạy B-16 còn 2026 chạy B-32,**cả hai đều 512 chiều** nên cái guard `qvec.shape[1] != index.d` trong UI khôngbắt được nhầm lẫn. Ghi rõ ra file là cách duy nhất chắc chắn.Sau khi chạy: **Save Version** → tải `features_*.npy` + `manifest_encode.json` về máy,đặt vào `data/processed_hcmc2026/`, rồi dựng lại FAISS index ở máy.

In [ ]:
man = {    "source": "kaggle_encode_corpus.ipynb",    "num_keyframes": int(len(paths)),    "shrink": "cạnh ngắn 384px JPEG q92 bicubic (đã đo: không tụt điểm)",    "row_order": "đọc thẳng metadata.parquet — sorted(video_id) rồi n tăng dần",    "models": {},}for name, pre in MODELS:    tag = f"{name}__{pre}".replace("/", "_")    f = OUT_DIR / f"features_{tag}.npy"    if f.exists():        a = np.load(f, mmap_mode="r")        man["models"][f"{name}/{pre}"] = {            "file": f.name, "dim": int(a.shape[1]), "dtype": str(a.dtype),            "final_score": results.get(f"{name}/{pre}"),            "size_mb": round(f.stat().st_size / 1024**2, 1),        }(OUT_DIR / "manifest_encode.json").write_text(    json.dumps(man, ensure_ascii=False, indent=2), encoding="utf-8")print(json.dumps(man, ensure_ascii=False, indent=2))# Kiểm tra lần cuối: vector phải đã chuẩn hoá, không NaN, không hàng toàn 0for name, pre in MODELS:    tag = f"{name}__{pre}".replace("/", "_")    f = OUT_DIR / f"features_{tag}.npy"    if not f.exists():        continue    a = np.load(f).astype(np.float32)    n = np.linalg.norm(a, axis=1)    print(f"{tag}: norm min {n.min():.4f} max {n.max():.4f} · "          f"NaN {int(np.isnan(a).sum())} · hàng toàn 0 {int((n < 1e-6).sum())}")